# RNN from Scratch
---
Now that we have learned all about the mechanisms of RNN and the mathematics behind it, lets build a simple forward-propagation RNN from scratch to understand more about them.

<p align="center">
 <img src="https://i.postimg.cc/25GXnjYK/RNNun-drawio.png" >
 <figcaption> Fig: Forward Propagation Recurrent Neural Network(RNN) (Unfolded)</figcaption>
</p>


We will take reference from Andrew Ngs **Deep Learning Specialization** course.

---
## Notations:

- $m$: batch size (number of sequences processed in parallel)  
- $T_x$: number of time steps (sequence length)  
- $n_x$: input size (features per time step)  
- $n_a$: hidden size (units in hidden state $a_t$)  
- $n_y$: output size (features/classes at each time step)

- $x_t \in \mathbb{R}^{n_x \times m}$: input at time $t$  
- $a_t \in \mathbb{R}^{n_a \times m}$: hidden state at time $t$  
- $y_t \in \mathbb{R}^{n_y \times m}$: output (logits/probs) at time $t$

Parameter shapes:
- $W_{ax} = \text{Wax} \in \mathbb{R}^{n_a \times n_x}$
- $W_{aa} = \text{Waa} \in \mathbb{R}^{n_a \times n_a}$
- $W_{ya} = \text{Wya} \in \mathbb{R}^{n_y \times n_a}$
- $b_a = \text{ba} \in \mathbb{R}^{n_a \times 1}$
- $b_y = \text{by} \in \mathbb{R}^{n_y \times 1}$


We are only going to use numpy for the simple RNN implementation. So, lets import numpy.

In [ ]:
import numpy as np

RNNs often output a probability distribution.

Thus, lets take the utility function **Softmax**, which takes raw scores (logits) and converts them into probabilities that sum to 1.

The standard formula for softmax is:
$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}, \quad i=1,\dots,K$$

where $z$ = the input logits

If $z$  has large positive values, $e^z$ can **overflow to inf**, which causes NaN values and instability/exploding gradients.

If $z$ has very negative values, $e^z$ can **underflow to 0**, which causes vanishing gradient problem.

RNNs process many time steps, so they repeatedly apply nonlinearities. That means logits can grow large or small quickly, making the instability worse.

To fix this, we subtract the maximum logit before exponentiation.

This doesn't change the probabilities (since adding/subtracting the same constant from all logits cancels out), but it keeps the numbers in a safe numerical range.

So, the stable formula we use is:

$$\text{softmax}(x)_i = \frac{e^{x_i - \max(x)}}{\sum_{j=1}^{K} e^{x_j - \max(x)}}, \quad i = 1, \dots, K$$


Now, lets initialize the function.

In [ ]:
def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0, keepdims=True)

Now, lets set up all the weights and biases needed for a simple RNN.

To initialize parameters, we will take the input:
- *n_x* : Input size (Number of features in your input vector *xt* at each time step)
- *n_a* : Hidden states size (Number of units in the RNN’s hidden state *at*)
- *n_y* : Output size (Number of features in your output vector *yt*)

Here, the output is a dictionary **params** that contains:    
  - *Wax* : Weight matrix multiplying the input, numpy array of shape (*n_a, n_x*)
  - *Waa* : Weight matrix multiplying the hidden state, numpy array of shape (*n_a, n_a*)
  - *Wya* : Weight matrix relating the hidden-state to the output, numpy array of shape (*n_y, n_a*)
  - *ba* :  Bias, numpy array of shape (*n_a, 1*)
  - *by* : Bias relating the hidden-state to the output, numpy array of shape (*n_y, 1*)

In [ ]:
def initialize_parameters(n_a, n_x, n_y):
    np.random.seed(1)
    Wax = np.random.randn(n_a, n_x) * 0.01  # input → hidden
    Waa = np.random.randn(n_a, n_a) * 0.01  # hidden → hidden
    Wya = np.random.randn(n_y, n_a) * 0.01  # hidden → output
    ba = np.zeros((n_a, 1))                  # hidden bias
    by = np.zeros((n_y, 1))                 # output bias

    params = {"Wax": Wax, "Waa": Waa, "Wya": Wya, "ba": ba, "by": by}
    return params


Now, lets implement a single forward RNN cell.

The function takes the following arguments:
  - *xt* : your input data at timestep "t", numpy array of shape (n_x, m).
  - *a_prev* : Hidden state at timestep "t-1", numpy array of shape (n_a, m)
  - *parameters* : python dictionary containing all the necessary weights and biases
    
    
The function returns:
- *a_next* : next hidden state, of shape (n_a, m)
- *yt_pred* : prediction at timestep "t", numpy array of shape (n_y, m)
- *cache* : tuple of values needed for the backward pass, contains (a_next, a_prev, xt, parameters)


In [ ]:
def rnn_cell_forward(xt, a_prev, parameters):

    # Retrieve parameters from "parameters"
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]

    # compute next activation state using the formula given before
    a_next = np.tanh(np.dot(Waa, a_prev) + np.dot(Wax, xt) + ba)
    # compute output of the current cell using the formula
    yt_pred = softmax(np.dot(Wya, a_next) + by)

    # store values you need for backward propagation in cache
    cache = (a_next, a_prev, xt, parameters)

    return a_next, yt_pred, cache

Now, lets set up dimensions, initialize parameters and implement the RNN forward cell.

In [ ]:
# Set dimensions
m = 10
n_x = 3
n_a = 5
n_y = 2

# Random input and previous hidden state
np.random.seed(1)
xt = np.random.randn(n_x, m)     # input at time t
a_prev = np.random.randn(n_a, m) # hidden state at t-1

# Initialize parameters
parameters = initialize_parameters(n_a, n_x, n_y)

# Forward step
a_next, yt_pred, cache = rnn_cell_forward(xt, a_prev, parameters)
print("a_next[4] = ", a_next[4])
print("a_next.shape = ", a_next.shape)
print("yt_pred[1] =", yt_pred[1])
print("yt_pred.shape = ", yt_pred.shape)

a_next[4] =  [-0.04360611  0.04901754  0.02861335 -0.00743636  0.01254669 -0.00387315
  0.01012874  0.00527268  0.03286051 -0.01785917]
a_next.shape =  (5, 10)
yt_pred[1] = [0.49978715 0.5002134  0.50009422 0.50011686 0.50011189 0.49965596
 0.50022647 0.50014628 0.50007505 0.49994581]
yt_pred.shape =  (2, 10)


## RNN forward pass

An RNN is the repetition of the cell you've just built.

If your input sequence of data is carried over 10 time steps, then you will copy the RNN cell 10 times.

<p align="center">
  <img src="https://i.postimg.cc/Xv6tWDNn/Unfold-RNN-drawio.png" alt="Unfolded RNN" width="800"/>
</p>
<p align="center">Figure 1. RNN(Unfolded)</p>

At a **single time step** (t), a batch input has shape (n_x, m).

When we **stack all time steps** (t = 0 to $T_{x-1}$) into one tensor, we keep:
- feature dimension \(n_x\),
- batch dimension \(m\),
- and add a **time dimension** (T_x).

Hence this function takes input data x of shape (n_x, m, T_x).

Each cell takes as input the hidden state from the previous cell and the current time-step's input data.

It outputs a hidden state and a prediction  for this time-step





The function takes following arguments:
- *x* : Input data for every time-step, of shape (*n_x, m, T_x*).
- *a0* : Initial hidden state, of shape (*n_a, m*)
- *parameters* : python dictionary containing all the necessary weights and biases

The function returns:
- *a* : Hidden states for every time-step, numpy array of shape (*n_a, m, T_x*)
- *y_pred* :Predictions for every time-step, numpy array of shape (*n_y, m, T_x*)
- *caches* : tuple of values needed for the backward pass, contains (*list of caches, x*)
    

### RNN Forward Pass (pseudocode)

For each time step $t = 1, 2, …, T$:

1. Compute the next hidden state  

$$
a^{\langle t \rangle} = \tanh(W_{ax} x^{\langle t \rangle} + W_{aa} a^{\langle t-1 \rangle} + b_a)
$$  

2. Compute the output prediction  

$$
\hat{y}^{\langle t \rangle} = \text{softmax}(W_{ya} a^{\langle t \rangle} + b_y)
$$  

3. Store $a^{\langle t \rangle}$ in `a[:,:,t]`  

4. Store $\hat{y}^{\langle t \rangle}$ in `y_pred[:,:,t]`  

5. Save *(a_next, a_prev, x_t, parameters)* as cache for backpropagation  


In [ ]:
def rnn_forward(x, a0, parameters):

    # Initialize "caches" which will contain the list of all caches
    caches = []

    # Retrieve dimensions from shapes of x and parameters["Wya"]
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wya"].shape

    # initialize "a" and "y" with zeros (≈2 lines)
    a = np.zeros((n_a, m, T_x))
    y_pred = np.zeros((n_y, m, T_x))

    # Initialize a_next (≈1 line)
    a_next = a0

    # loop over all time-steps
    for t in range(T_x):
        # Update next hidden state, compute the prediction, get the cache
        a_next, yt_pred, cache = rnn_cell_forward(x[:,:,t], a_next, parameters)
        # Save the value of the new "next" hidden state in a
        a[:,:,t] = a_next
        # Save the value of the prediction in y
        y_pred[:,:,t] = yt_pred
        # Append "cache" to "caches"
        caches.append(cache)

    # store values needed for backward propagation in cache
    caches = (caches, x)

    return a, y_pred, caches

Here, lets initialize the dimension T_x, initialize weights and biases, and implement the forward propagation RNN.

In [ ]:
# Define dimensions
T_x = 4   # sequence length

# Random inputs
np.random.seed(1)
x = np.random.randn(n_x, m, T_x)
a0 = np.random.randn(n_a, m)

# Initialize parameters using your function
parameters = initialize_parameters(n_a, n_x, n_y)

# Forward pass
a, y_pred, caches = rnn_forward(x, a0, parameters)
print("a[4][1] = ", a[4][1])
print("a.shape = ", a.shape)
print("y_pred[1][3] =", y_pred[1][3])
print("y_pred.shape = ", y_pred.shape)
print("caches[1][1][3] =", caches[1][1][3])
print("len(caches) = ", len(caches))

a[4][1] =  [-0.04191705  0.01183832 -0.01423173 -0.0011517 ]
a.shape =  (5, 10, 4)
y_pred[1][3] = [0.50006689 0.49995976 0.50013891 0.4998614 ]
y_pred.shape =  (2, 10, 4)
caches[1][1][3] = [-1.1425182  -0.34934272 -0.20889423  0.58662319]
len(caches) =  2


## Conclusion

- We implemented an **RNN forward pass** with clean shapes and stable softmax.


- The RNN cell update is:  
  $$
  a_t = \tanh(W_{aa} a_{t-1} + W_{ax} x_t + b_a), \quad
  \hat{y}_t = \text{softmax}(W_{ya} a_t + b_y)
  $$
- Using variables for dimensions $(n_x, n_a, n_y, m, T_x)$ improves **readability and reproducibility**.
- This sets the stage for:
  - **Backpropagation Through Time (BPTT)**
  - **Gated units** (GRU/LSTM) to handle long-term dependencies
  - **Regularization** (dropout, clipping) to reduce overfitting/instability
